# Ribo-seq / RNA-seq Integrated Analysis for 3'UTR Design

**Dataset:** GEO Series GSE61334  
**Reference:** GRCh38.p14 (GENCODE v46)

| SRA ID | GEO ID | Sample | Type |
|--------|--------|--------|------|
| SRR1562541 | GSM1495246 | Tumor Ribo-seq A | riboseq |
| SRR1562542 | GSM1495247 | Normal Ribo-seq A | riboseq |
| SRR1562543 | GSM1495248 | Tumor Ribo-seq B | riboseq |
| SRR1562544 | GSM1495249 | Normal RNA-seq A | rnaseq |
| SRR1562545 | GSM1495250 | Normal RNA-seq B | rnaseq |
| SRR1562546 | GSM1495251 | Normal RNA-seq C | rnaseq |
| SRR1562547 | GSM1495252 | Tumor RNA-seq A | rnaseq |
| SRR1562548 | GSM1495253 | Tumor RNA-seq B | rnaseq |

**Research goal:** Design custom RNA sequences with optimized 3'UTRs for protein production. This notebook extracts translational efficiency patterns and 3'UTR sequence features from matched Ribo-seq/RNA-seq data to identify design rules for synthetic constructs.

## Environment Setup
```bash
conda env create -f environment.yml
conda activate patrick_rnaseq
python -m ipykernel install --user --name patrick_rnaseq --display-name "Patrick RNA-seq"
```

## 1. Setup and Configuration

This cell imports all required libraries and defines the analysis parameters. Key design choices:

- **Paths** are parameterized so the notebook can be adapted to other datasets by changing variables at the top
- **Sample metadata** includes condition labels (tumor/normal) for per-condition TE analysis
- **Visualization** uses the Okabe-Ito colorblind-safe palette throughout, following publication best practices from the scientific-visualization skill
- **`find_fastq()`** handles the mixed filename formats in this dataset (some files have GEO descriptors, others have bare SRA IDs)

In [ ]:
import json, re, subprocess, gzip
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Parameterized paths ---
BASE = Path("/Volumes/Felix_SSD/Patrick")
GENCODE_RELEASE = "46"
MIN_MAPQ = 10
MIN_READS_TE = 50
N_UTR_GENES = 500

REF_DIR = BASE / "reference"
STAR_INDEX = REF_DIR / "STAR_index"
GENOME_FA = REF_DIR / "GRCh38.primary_assembly.genome.fa"
GTF = REF_DIR / f"gencode.v{GENCODE_RELEASE}.annotation.gtf"
RRNA_DIR = REF_DIR / "rRNA"
TRIM_DIR = BASE / "trimmed"
DEPLETED_DIR = BASE / "rRNA_depleted"
ALIGN_DIR = BASE / "aligned"
BIGWIG_DIR = BASE / "bigwig"

for d in [REF_DIR, RRNA_DIR, TRIM_DIR, DEPLETED_DIR, ALIGN_DIR, BIGWIG_DIR]:
    d.mkdir(exist_ok=True)

# --- Sample metadata (resolved from GEO series GSE61334) ---
SAMPLES = {
    "SRR1562541": {"name": "Tumor Ribo-seq A",  "type": "riboseq", "condition": "tumor"},
    "SRR1562542": {"name": "Normal Ribo-seq A", "type": "riboseq", "condition": "normal"},
    "SRR1562543": {"name": "Tumor Ribo-seq B",  "type": "riboseq", "condition": "tumor"},
    "SRR1562544": {"name": "Normal RNA-seq A",  "type": "rnaseq",  "condition": "normal"},
    "SRR1562545": {"name": "Normal RNA-seq B",  "type": "rnaseq",  "condition": "normal"},
    "SRR1562546": {"name": "Normal RNA-seq C",  "type": "rnaseq",  "condition": "normal"},
    "SRR1562547": {"name": "Tumor RNA-seq A",   "type": "rnaseq",  "condition": "tumor"},
    "SRR1562548": {"name": "Tumor RNA-seq B",   "type": "rnaseq",  "condition": "tumor"},
}

def find_fastq(sra_id):
    for p in BASE.glob("*.fastq"):
        if sra_id in p.name: return p
    return None

# --- Plotly publication template (Okabe-Ito colorblind-safe) ---
OKABE_ITO = ["#E69F00","#56B4E9","#009E73","#F0E442","#0072B2","#D55E00","#CC79A7","#000000"]
SAMPLE_COLORS = {sra: OKABE_ITO[i] for i, sra in enumerate(SAMPLES)}
TYPE_COLORS = {"riboseq": "#56B4E9", "rnaseq": "#E69F00"}
COND_COLORS = {"tumor": "#D55E00", "normal": "#009E73"}

def apply_publication_style(fig, width=1000, height=600, x_grid=False, y_grid=True, font_size=13):
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", width=width, height=height,
        font=dict(family="Arial, Helvetica, sans-serif", size=font_size, color="black"),
        margin=dict(l=80, r=40, t=60, b=60))
    ax = dict(showline=True, linewidth=2, linecolor="black", tickcolor="black", ticks="outside")
    fig.update_xaxes(**ax, showgrid=x_grid, gridcolor="lightgray")
    fig.update_yaxes(**ax, showgrid=y_grid, gridcolor="lightgray")
    return fig

def set_legend_position(fig, position="top-right"):
    p = {"top-left":(0.99,0.01,"top","left"),"top-right":(0.99,0.99,"top","right"),
         "bottom-left":(0.01,0.01,"bottom","left"),"bottom-right":(0.01,0.99,"bottom","right")}
    y,x,ya,xa = p.get(position, p["top-right"])
    fig.update_layout(legend=dict(yanchor=ya,y=y,xanchor=xa,x=x,bgcolor="rgba(255,255,255,0.8)",borderwidth=0))
    return fig

# --- Gene ID to symbol mapping ---
def build_gene_map(gtf_path):
    gmap = {}
    for line in open(gtf_path):
        if line.startswith("#"): continue
        f = line.split("\t")
        if f[2] != "gene": continue
        gid = re.search(r'gene_id "([^"]+)"', f[8])
        gn = re.search(r'gene_name "([^"]+)"', f[8])
        if gid and gn:
            gmap[gid.group(1).split(".")[0]] = gn.group(1)
    return gmap

if GTF.exists():
    GENE_MAP = build_gene_map(GTF)
    print(f"Gene map: {len(GENE_MAP):,} genes")

for sra, m in SAMPLES.items():
    fq = find_fastq(sra)
    print(f"  {sra} [{m['type']:7s}] {m['condition']:6s} {m['name']:25s} {'FOUND' if fq else 'MISSING'}")

## 2. Reference Genome Download and Indexing

**What:** Download the human reference genome (GRCh38.p14) and gene annotation (GENCODE v46), then build a STAR genome index.

**Why:** All sequenced reads must be mapped back to the genome to determine which gene they came from. STAR is a splice-aware aligner that handles both the short Ribo-seq reads (~28 nt) and longer RNA-seq reads (101 nt). The GENCODE annotation tells us where each gene, exon, CDS, and UTR is located.

**How:** STAR pre-builds a suffix array index from the genome + annotation. This takes ~30 min and ~32 GB RAM but only needs to be done once. The `--sjdbOverhang 100` parameter tells STAR the maximum read overhang for splice junction detection.

In [ ]:
GENCODE = f"https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_{GENCODE_RELEASE}"
for local, url in {GENOME_FA.with_suffix(".fa.gz"): f"{GENCODE}/GRCh38.primary_assembly.genome.fa.gz",
                   GTF.with_suffix(".gtf.gz"): f"{GENCODE}/gencode.v{GENCODE_RELEASE}.annotation.gtf.gz"}.items():
    if local.exists() or local.with_suffix("").exists(): print(f"  Exists: {local.name}"); continue
    print(f"  Downloading {local.name}...")
    subprocess.run(["curl","-L","-o",str(local),url], check=True)
for gz in [GENOME_FA.with_suffix(".fa.gz"), GTF.with_suffix(".gtf.gz")]:
    dec = gz.with_suffix("")
    if not dec.exists() and gz.exists():
        print(f"  Decompressing {gz.name}..."); subprocess.run(["gunzip","-k",str(gz)], check=True)
    else: print(f"  Ready: {dec.name}")

In [ ]:
STAR_INDEX.mkdir(exist_ok=True)
if (STAR_INDEX / "SA").exists(): print("STAR index exists.")
else:
    print("Building STAR index (~30 min)...")
    r = subprocess.run(["STAR","--runMode","genomeGenerate","--genomeDir",str(STAR_INDEX),
        "--genomeFastaFiles",str(GENOME_FA),"--sjdbGTFfile",str(GTF),
        "--runThreadN","8","--sjdbOverhang","100"], capture_output=True, text=True)
    if r.returncode!=0: raise RuntimeError(f"STAR index failed: {r.stderr[-500:]}")
    print("Done.")

## 3. Adapter Trimming (fastp)

**What:** Remove sequencing adapter sequences from the raw reads and filter by length.

**Why:** Illumina sequencing reads through the insert and into the adapter on the other side. For Ribo-seq, the insert (ribosome-protected fragment, RPF) is only ~28 nt, but reads are sequenced at 101 cycles, so ~73 nt of each read is adapter. Adapter sequences must be removed before alignment, or reads will fail to map.

**How:** fastp auto-detects the adapter sequence and trims it. For Ribo-seq, we additionally filter by length:
- **Minimum 20 nt**: Reads shorter than this cannot be uniquely mapped
- **Maximum 35 nt**: Genuine RPFs are 26-34 nt; longer fragments are likely non-ribosomal contamination

RNA-seq reads are full-length (101 nt) with minimal adapter contamination, so only the 20 nt minimum filter applies.

In [ ]:
def run_fastp(sra_id, stype):
    out = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if out.exists(): print(f"  {sra_id}: exists"); return
    fq = find_fastq(sra_id)
    if not fq: print(f"  {sra_id}: not found"); return
    cmd = ["fastp","-i",str(fq),"-o",str(out),"--html",str(TRIM_DIR/f"{sra_id}_fastp.html"),
           "--json",str(TRIM_DIR/f"{sra_id}_fastp.json"),"--thread","8","--length_required","20"]
    if stype=="riboseq": cmd+=["--length_limit","35"]
    print(f"  {sra_id}: trimming...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  {sra_id}: {'done' if r.returncode==0 else 'ERROR'}")

for s,m in SAMPLES.items(): run_fastp(s, m["type"])

## 4. rRNA Removal (bowtie2)

**What:** Remove ribosomal RNA reads from all samples before genome alignment.

**Why:** During Ribo-seq library preparation, ribosomes are purified by sucrose gradient centrifugation. The ribosome itself is ~60% rRNA by mass, so ribosomal RNA fragments massively contaminate the library (typically 50-90% of reads). If left in, these rRNA reads:
- Inflate the total read count denominator in CPM/RPM normalization
- Artificially compress the normalized expression values of all protein-coding genes
- Distort translational efficiency calculations

RNA-seq libraries (poly-A selected or ribo-depleted) also have some residual rRNA contamination (1-15%), so we filter all samples.

**How:** We align trimmed reads against the four human cytoplasmic rRNA sequences using bowtie2, and retain only the **unmapped** reads (non-rRNA) for downstream analysis.

**Database:** NCBI RefSeq human ribosomal RNA:
- 5S rRNA (NR_023363) - 121 nt, part of the large 60S subunit
- 5.8S rRNA (NR_003285) - 157 nt, part of the large 60S subunit
- 18S rRNA (NR_003286) - 1,869 nt, the small 40S subunit RNA
- 28S rRNA (NR_003287) - 5,070 nt, the large 60S subunit RNA

In [ ]:
# Download human rRNA sequences and build bowtie2 index
rrna_fa = RRNA_DIR / "human_rRNA.fa"
rrna_idx = RRNA_DIR / "human_rRNA"

if not rrna_fa.exists():
    print("Downloading human rRNA sequences from NCBI RefSeq...")
    rrna_accs = ["NR_023363.1", "NR_003285.2", "NR_003286.4", "NR_003287.4"]
    with open(rrna_fa, "w") as out:
        for acc in rrna_accs:
            r = subprocess.run(["curl","-s",
                f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nucleotide&id={acc}&rettype=fasta&retmode=text"],
                capture_output=True, text=True)
            out.write(r.stdout)
    print(f"  Wrote {rrna_fa.name}")
else: print("rRNA FASTA exists.")

if not (RRNA_DIR / "human_rRNA.1.bt2").exists():
    print("Building bowtie2 rRNA index...")
    r = subprocess.run(["bowtie2-build", str(rrna_fa), str(rrna_idx)], capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-300:]}")
else: print("rRNA index exists.")

### rRNA filtering

For each sample, reads are aligned to the rRNA index. Reads that **do not** align (the non-rRNA fraction) are saved for genome alignment. The rRNA contamination percentage is reported as a key QC metric.

In [ ]:
rrna_stats = []
for sra_id, meta in SAMPLES.items():
    input_fq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    output_fq = DEPLETED_DIR / f"{sra_id}_no_rRNA.fastq.gz"
    if output_fq.exists():
        print(f"  {sra_id}: rRNA-depleted file exists"); continue
    if not input_fq.exists():
        print(f"  {sra_id}: trimmed FASTQ not found"); continue

    print(f"  {sra_id}: removing rRNA ({meta['type']})...")
    r = subprocess.run([
        "bowtie2", "-x", str(RRNA_DIR / "human_rRNA"),
        "-U", str(input_fq), "--un-gz", str(output_fq),
        "-S", "/dev/null", "-p", "8", "--very-fast",
    ], capture_output=True, text=True)
    for line in r.stderr.splitlines():
        if "overall alignment rate" in line:
            pct = line.strip().split()[0]
            rrna_stats.append({"Sample": meta["name"], "SRA": sra_id, "rRNA (%)": pct})
            print(f"  {sra_id}: {pct} rRNA")

if rrna_stats:
    print("\nrRNA contamination summary:")
    for s in rrna_stats: print(f"  {s['Sample']}: {s['rRNA (%)']}")

## 5. QC Summary

**What:** Parse the fastp trimming reports to compare quality metrics across all samples.

**Why:** Before investing compute time in alignment, we verify that:
- Read counts are sufficient for downstream analysis (>10M reads per sample)
- Adapter detection worked (high adapter-trimmed % for Ribo-seq)
- Duplication rates are in expected ranges (Ribo-seq: 40-70% is normal due to short fragments + deep sequencing)
- GC content is consistent (human genome ~40% GC; post-trim Ribo-seq may show elevated GC from residual rRNA)

In [ ]:
qc_rows = []
for sra_id, meta in SAMPLES.items():
    jp = TRIM_DIR / f"{sra_id}_fastp.json"
    if not jp.exists(): continue
    rpt = json.loads(jp.read_text())
    b, a, f = rpt["summary"]["before_filtering"], rpt["summary"]["after_filtering"], rpt["filtering_result"]
    qc_rows.append({"Sample": meta["name"], "SRA": sra_id, "Type": meta["type"],
        "Condition": meta["condition"],
        "Raw reads (M)": b["total_reads"]/1e6, "Passed reads (M)": a["total_reads"]/1e6,
        "Pass rate (%)": f["passed_filter_reads"]/b["total_reads"]*100,
        "Adapter trimmed (M)": rpt.get("adapter_cutting",{}).get("adapter_trimmed_reads",0)/1e6,
        "Duplication (%)": rpt.get("duplication",{}).get("rate",0)*100,
        "GC (%)": b["gc_content"]*100})
qc_df = pd.DataFrame(qc_rows)
qc_df

### QC visualization

Three-panel bar chart comparing read counts, duplication rates, and GC content across all 8 samples. Raw reads (gray) vs reads passing filters (colored) shows how much data is retained after trimming.

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Read counts","Duplication rate","GC content"), horizontal_spacing=0.08)
colors = [SAMPLE_COLORS[s] for s in qc_df["SRA"]]
fig.add_trace(go.Bar(name="Raw", x=qc_df["Sample"], y=qc_df["Raw reads (M)"], marker_color="lightgray"), row=1, col=1)
fig.add_trace(go.Bar(name="Passed", x=qc_df["Sample"], y=qc_df["Passed reads (M)"], marker_color=colors), row=1, col=1)
fig.add_trace(go.Bar(x=qc_df["Sample"], y=qc_df["Duplication (%)"], marker_color=colors, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=qc_df["Sample"], y=qc_df["GC (%)"], marker_color=colors, showlegend=False), row=1, col=3)
fig.update_layout(barmode="group")
fig.update_yaxes(title_text="Reads (millions)", row=1, col=1)
fig.update_yaxes(title_text="Duplication (%)", row=1, col=2)
fig.update_yaxes(title_text="GC (%)", row=1, col=3)
fig.update_xaxes(tickangle=-40)
apply_publication_style(fig, width=1400, height=500); set_legend_position(fig, "top-left")
fig.show()

### MultiQC report

MultiQC aggregates all fastp and STAR reports into a single interactive HTML document, suitable for supplementary material.

In [ ]:
multiqc_dir = BASE / "multiqc_report"
if not multiqc_dir.exists():
    print("Running MultiQC...")
    r = subprocess.run(["multiqc", str(TRIM_DIR), str(ALIGN_DIR),
        "-o", str(multiqc_dir), "--force", "--quiet"], capture_output=True, text=True)
    if r.returncode == 0: print(f"Report: {multiqc_dir / 'multiqc_report.html'}")
    else: print("MultiQC not available (install: conda install -c bioconda multiqc)")
else: print(f"Report exists: {multiqc_dir / 'multiqc_report.html'}")

### Ribo-seq read length QC

**Why this matters for your designs:** The read length distribution after trimming directly reflects the size of ribosome-protected fragments (RPFs). Genuine RPFs from eukaryotic 80S ribosomes should peak at 28-31 nt. If the peak is shorter or the distribution is flat, the library may contain degradation products rather than true ribosome footprints, which would undermine all downstream TE and coverage analyses.

In [ ]:
for sra_id, meta in SAMPLES.items():
    if meta["type"] != "riboseq": continue
    jp = TRIM_DIR / f"{sra_id}_fastp.json"
    if not jp.exists(): continue
    rpt = json.loads(jp.read_text())
    passed = rpt["summary"]["after_filtering"]["total_reads"]
    total_bases = rpt["summary"]["after_filtering"]["total_bases"]
    avg_len = total_bases / passed if passed > 0 else 0
    print(f"  {meta['name']}: avg RPF length = {avg_len:.1f} nt, {passed:,} reads")

print("\nExpected: peak at 28-31 nt. Lengths outside 26-34 nt may indicate non-RPF contamination.")
print("For detailed length distributions, see the fastp HTML reports in trimmed/.")

## 6. STAR Alignment

**What:** Map the cleaned reads to the human genome to determine which gene each read came from.

**Why:** Each sequenced fragment (read) is a short piece of RNA. By finding where it aligns in the genome, we can assign it to a specific gene, exon, or UTR. STAR is used because it can handle both:
- **Ribo-seq** (short 28 nt unspliced reads): `--alignEndsType EndToEnd` prevents soft-clipping of these already short reads; `--alignIntronMax 1` disables spliced alignment since RPFs almost never span exon junctions
- **RNA-seq** (101 nt spliced reads): default splice-aware mode handles intron-spanning reads

`--quantMode GeneCounts` tells STAR to also count reads per gene during alignment, which we use later for QC.

**Run time:** ~10-20 min per sample. Samples run **sequentially** because STAR loads the entire genome index (~30 GB) into shared memory.

In [ ]:
def run_star(sra_id, stype):
    bam = ALIGN_DIR / f"{sra_id}_Aligned.sortedByCoord.out.bam"
    if bam.exists() and bam.stat().st_size > 0:
        print(f"  {sra_id}: BAM exists ({bam.stat().st_size/1e9:.1f} GB)"); return
    fq = DEPLETED_DIR / f"{sra_id}_no_rRNA.fastq.gz"
    if not fq.exists():
        fq = TRIM_DIR / f"{sra_id}_trimmed.fastq.gz"
    if not fq.exists(): print(f"  {sra_id}: no FASTQ"); return
    cmd = ["STAR","--runThreadN","8","--genomeDir",str(STAR_INDEX),"--readFilesIn",str(fq),
           "--readFilesCommand","zcat","--outSAMtype","BAM","SortedByCoordinate",
           "--outFileNamePrefix",str(ALIGN_DIR/f"{sra_id}_"),"--quantMode","GeneCounts",
           "--outSAMattributes","NH","HI","AS","NM","MD"]
    if stype=="riboseq":
        cmd+=["--alignIntronMax","1","--outFilterMismatchNmax","2",
              "--outFilterMultimapNmax","1","--alignEndsType","EndToEnd"]
    print(f"  {sra_id}: aligning ({stype})...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode!=0: raise RuntimeError(f"STAR failed for {sra_id}: {r.stderr[-500:]}")
    r2 = subprocess.run(["samtools","index",str(bam)], capture_output=True, text=True)
    if r2.returncode!=0: print(f"  WARNING: samtools index failed for {sra_id}")
    print(f"  {sra_id}: done ({bam.stat().st_size/1e9:.1f} GB)")

for s,m in SAMPLES.items(): run_star(s, m["type"])

### Alignment statistics

The stacked bar chart shows the fraction of reads that mapped uniquely, mapped to multiple locations (multi-mapped), or failed to map. For Ribo-seq, uniquely mapped rates of 50-80% are typical (many reads are lost to rRNA remnants or repetitive sequences). For RNA-seq, >80% unique mapping is expected.

In [ ]:
def parse_star_log(sid):
    lp = ALIGN_DIR / f"{sid}_Log.final.out"
    if not lp.exists(): return None
    stats = {}
    for line in lp.read_text().splitlines():
        if "|" in line:
            k, v = line.split("|",1); k, v = k.strip(), v.strip().rstrip("%")
            try: stats[k] = float(v)
            except ValueError: stats[k] = v
    return stats

align_rows = []
for sid, meta in SAMPLES.items():
    s = parse_star_log(sid)
    if not s: continue
    align_rows.append({"Sample": meta["name"], "SRA": sid, "Type": meta["type"],
        "Input reads (M)": s.get("Number of input reads",0)/1e6,
        "Uniquely mapped (%)": s.get("Uniquely mapped reads %",0),
        "Multi-mapped (%)": s.get("% of reads mapped to multiple loci",0),
        "Unmapped too short (%)": s.get("% of reads unmapped: too short",0)})
align_df = pd.DataFrame(align_rows)

fig = go.Figure()
for col, color in [("Uniquely mapped (%)","#009E73"),("Multi-mapped (%)","#E69F00"),("Unmapped too short (%)","#D55E00")]:
    fig.add_trace(go.Bar(name=col.replace(" (%)",""), x=align_df["Sample"], y=align_df[col], marker_color=color))
fig.update_layout(barmode="stack", title="STAR alignment rates", yaxis_title="Reads (%)", yaxis_range=[0,105], xaxis_tickangle=-30)
apply_publication_style(fig); set_legend_position(fig, "top-right")
fig.show()
align_df

### Strandedness auto-detection

**What:** Determine whether the sequencing libraries are strand-specific by comparing sense vs antisense read counts from STAR's per-gene output.

**Why:** If the library is stranded but we count reads as unstranded (`-s 0`), we'll erroneously include antisense reads, inflating counts for ~15-20% of genes that have overlapping genes on the opposite strand. This directly corrupts TE values.

In [ ]:
print("Strandedness detection (from STAR ReadsPerGene):")
print(f"  {'Sample':<30s} {'Sense':>12s} {'Antisense':>12s} {'Ratio':>8s} {'Verdict'}")
strand_verdicts = []
for sid, meta in SAMPLES.items():
    rpg = ALIGN_DIR / f"{sid}_ReadsPerGene.out.tab"
    if not rpg.exists(): continue
    df = pd.read_csv(rpg, sep="\t", header=None, skiprows=4, names=["gene","unstranded","sense","antisense"])
    sense_total = df["sense"].sum()
    anti_total = df["antisense"].sum()
    ratio = sense_total / max(anti_total, 1)
    if ratio > 5: verdict = "stranded (sense, -s 1)"
    elif 1/ratio > 5: verdict = "stranded (antisense, -s 2)"
    else: verdict = "unstranded (-s 0)"
    strand_verdicts.append(verdict)
    print(f"  {meta['name']:<30s} {sense_total:>12,} {anti_total:>12,} {ratio:>8.1f} {verdict}")

if any("stranded" in v and "-s 0" not in v for v in strand_verdicts):
    print("\n  WARNING: Stranded libraries detected. Consider re-running featureCounts with the appropriate -s flag.")

## 7. Ribo-seq Quality Control

**What:** Verify that the Ribo-seq data genuinely represents translating ribosomes through two orthogonal checks: sample-level clustering (PCA) and reading frame analysis (3-nt periodicity).

**Why these checks are essential:**

1. **PCA** reveals whether samples group by assay type (Ribo-seq vs RNA-seq) and condition (tumor vs normal). If a sample clusters with the wrong group, it may be mislabeled or compromised.

2. **3-nt periodicity** is the defining signature of genuine ribosome-protected fragments. Ribosomes move along mRNA in steps of 3 nucleotides (one codon at a time). If the sequenced fragments truly come from translating ribosomes, the 5' ends of RPFs should show a strong bias toward reading frame 0 (the frame that matches the annotated CDS). Random RNA degradation products show no frame preference (~33% in each frame).

**For your designs:** If the RPFs lack periodicity, the TE values and coverage patterns cannot be trusted as reflecting genuine translation.

In [ ]:
# PCA on all samples
sample_counts = {}
for sid in SAMPLES:
    rpg = ALIGN_DIR / f"{sid}_ReadsPerGene.out.tab"
    if not rpg.exists(): continue
    df = pd.read_csv(rpg, sep="\t", header=None, skiprows=4, names=["gene","unstranded","sense","antisense"])
    sample_counts[sid] = df.set_index("gene")["unstranded"]

if sample_counts:
    count_matrix = pd.DataFrame(sample_counts).fillna(0)
    count_matrix = count_matrix.loc[count_matrix.sum(axis=1) > 10]
    log2_counts = np.log2(count_matrix + 1)

    pca = PCA(n_components=2)
    coords = pca.fit_transform(log2_counts.T)
    pca_df = pd.DataFrame(coords, columns=["PC1","PC2"], index=count_matrix.columns)
    pca_df["Sample"] = [SAMPLES[s]["name"] for s in pca_df.index]
    pca_df["Type"] = [SAMPLES[s]["type"] for s in pca_df.index]
    pca_df["Condition"] = [SAMPLES[s]["condition"] for s in pca_df.index]

    fig = px.scatter(pca_df, x="PC1", y="PC2", color="Type", symbol="Condition",
        text="Sample", color_discrete_map=TYPE_COLORS,
        labels={"PC1": f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
                "PC2": f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)"})
    fig.update_traces(textposition="top center", marker=dict(size=12))
    fig.update_layout(title="PCA of gene counts")
    apply_publication_style(fig, width=800, height=600)
    fig.show()

    # Spearman correlation heatmap
    corr = log2_counts.corr(method="spearman")
    corr.index = [SAMPLES[s]["name"] for s in corr.index]
    corr.columns = [SAMPLES[s]["name"] for s in corr.columns]
    fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=0.5, zmax=1,
        text_auto=".2f", title="Spearman correlation (log2 counts)")
    apply_publication_style(fig, width=800, height=700)
    fig.show()
else:
    print("Run alignment first.")

### Reading frame analysis (3-nt periodicity)

**What:** For each Ribo-seq sample, count how many CDS-mapping reads fall in each of the three reading frames. Genuine RPFs should show >45% of reads in frame 0 (the annotated ORF frame).

**How:** For each CDS region, we calculate `(read_5prime_position - CDS_start) % 3` to determine the frame. This is done on a subset of 500 genes for speed.

In [ ]:
import pysam

def check_frame(bam_path, gtf_path, n_genes=500):
    cds_regions = []
    for line in open(gtf_path):
        if line.startswith("#"): continue
        f = line.strip().split("\t")
        if f[2] != "CDS": continue
        gid = re.search(r'gene_id "([^"]+)"', f[8])
        if gid:
            cds_regions.append((f[0], int(f[3])-1, int(f[4]), f[6], gid.group(1).split(".")[0]))
    cds_df = pd.DataFrame(cds_regions, columns=["chrom","start","end","strand","gene_id"])
    cds_df = cds_df.drop_duplicates("gene_id").head(n_genes)

    frame_counts = [0, 0, 0]
    bam = pysam.AlignmentFile(str(bam_path), "rb")
    for _, row in cds_df.iterrows():
        try:
            for read in bam.fetch(row["chrom"], row["start"], row["end"]):
                if read.is_unmapped or read.mapping_quality < MIN_MAPQ: continue
                pos = read.reference_start
                if row["strand"] == "+":
                    frame = (pos - row["start"]) % 3
                else:
                    frame = (row["end"] - read.reference_end) % 3
                frame_counts[frame] += 1
        except ValueError: continue
    bam.close()
    total = sum(frame_counts)
    if total == 0: return [0, 0, 0]
    return [c/total*100 for c in frame_counts]

frame_rows = []
for sid, meta in SAMPLES.items():
    if meta["type"] != "riboseq": continue
    bam = ALIGN_DIR / f"{sid}_Aligned.sortedByCoord.out.bam"
    if not bam.exists() or bam.stat().st_size == 0: continue
    print(f"  {meta['name']}: checking frame distribution...")
    frames = check_frame(bam, GTF)
    frame_rows.append({"Sample": meta["name"], "Frame 0 (%)": frames[0],
                       "Frame 1 (%)": frames[1], "Frame 2 (%)": frames[2]})
    print(f"    Frame 0: {frames[0]:.1f}% | Frame 1: {frames[1]:.1f}% | Frame 2: {frames[2]:.1f}%")

if frame_rows:
    frame_df = pd.DataFrame(frame_rows)
    fig = go.Figure()
    for i, (col, color) in enumerate([("Frame 0 (%)","#009E73"),("Frame 1 (%)","#E69F00"),("Frame 2 (%)","#D55E00")]):
        fig.add_trace(go.Bar(name=f"Frame {i}", x=frame_df["Sample"], y=frame_df[col], marker_color=color))
    fig.update_layout(barmode="group", title="Reading frame distribution (Ribo-seq CDS reads)", yaxis_title="Reads (%)")
    fig.add_hline(y=33.3, line_dash="dash", line_color="gray", annotation_text="Random (33%)")
    apply_publication_style(fig, height=400)
    fig.show()
    print("Frame 0 should be >45% for genuine RPFs (vs 33% random).")

## 8. Coverage Tracks (deepTools bamCoverage)

**What:** Convert BAM alignment files into normalized coverage tracks (bigWig format) using deepTools.

**Why:** bigWig files store per-position read coverage across the genome in a compressed, indexed format. They are used by `computeMatrix` in Section 11 to efficiently compute coverage across thousands of 3'UTR regions simultaneously. CPM (counts per million) normalization makes coverage values comparable between samples with different sequencing depths.

**MAPQ filter:** `--minMappingQuality 10` excludes multi-mapped reads from the coverage tracks, ensuring that signal at each position comes from uniquely placed reads.

In [ ]:
for sid in SAMPLES:
    bam = ALIGN_DIR / f"{sid}_Aligned.sortedByCoord.out.bam"
    bw = BIGWIG_DIR / f"{sid}.bw"
    if bw.exists(): print(f"  {sid}: exists"); continue
    if not bam.exists() or bam.stat().st_size==0: print(f"  {sid}: no BAM"); continue
    print(f"  {sid}: bamCoverage...")
    r = subprocess.run(["bamCoverage","--bam",str(bam),"--outFileName",str(bw),
        "--normalizeUsing","CPM","--binSize","10","--numberOfProcessors","8",
        "--minMappingQuality",str(MIN_MAPQ)], capture_output=True, text=True)
    print(f"  {sid}: {'done' if r.returncode==0 else 'ERROR'}")

## 9. Gene-Level Counting (featureCounts)

**What:** Count how many reads fall within each gene for every sample.

**Why this uses different feature types for Ribo-seq vs RNA-seq:**

- **Ribo-seq counts over CDS only** (`-t CDS`): Translational efficiency measures ribosome density on the coding sequence. Including UTR reads would inflate the Ribo-seq numerator and create a circular artifact: genes with 3'UTR read-through would appear to have higher TE simply because their 3'UTR RPFs are counted as part of the gene total.

- **RNA-seq counts over exons** (`-t exon`): mRNA abundance includes the full transcript (5'UTR + CDS + 3'UTR), so counting all exonic reads gives the correct denominator for TE.

**MAPQ filter:** `--minMQS 10` excludes multi-mapped reads for consistent counting.

In [ ]:
ribo_counts_file = ALIGN_DIR / "ribo_cds_counts.txt"
rnaseq_counts_file = ALIGN_DIR / "rnaseq_exon_counts.txt"

ribo_bams = [str(ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam") for s in SAMPLES
             if SAMPLES[s]["type"]=="riboseq"
             and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").exists()
             and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").stat().st_size>0]
rnaseq_bams = [str(ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam") for s in SAMPLES
               if SAMPLES[s]["type"]=="rnaseq"
               and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").exists()
               and (ALIGN_DIR/f"{s}_Aligned.sortedByCoord.out.bam").stat().st_size>0]

if not ribo_counts_file.exists() and ribo_bams:
    print(f"featureCounts (CDS) on {len(ribo_bams)} Ribo-seq BAMs...")
    r = subprocess.run(["featureCounts","-a",str(GTF),"-o",str(ribo_counts_file),"-T","8",
        "-t","CDS","-g","gene_id","--primary","-s","0","--minMQS",str(MIN_MAPQ)]+ribo_bams,
        capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")

if not rnaseq_counts_file.exists() and rnaseq_bams:
    print(f"featureCounts (exon) on {len(rnaseq_bams)} RNA-seq BAMs...")
    r = subprocess.run(["featureCounts","-a",str(GTF),"-o",str(rnaseq_counts_file),"-T","8",
        "-t","exon","-g","gene_id","--primary","-s","0","--minMQS",str(MIN_MAPQ)]+rnaseq_bams,
        capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")

### Parse and visualize gene counts

Gene IDs are mapped to human-readable gene symbols using the GENCODE GTF annotation.

In [ ]:
def parse_fc(path, sample_type):
    raw = pd.read_csv(path, sep="\t", comment="#")
    col_map = {c: s for c in raw.columns for s in SAMPLES if s in c}
    raw = raw.rename(columns=col_map)
    raw["gene_id"] = raw["Geneid"].str.split(".").str[0]
    sra_ids = [s for s in SAMPLES if s in raw.columns and SAMPLES[s]["type"]==sample_type]
    df = raw.set_index("gene_id")[sra_ids].copy()
    df.columns = [SAMPLES[s]["name"] for s in sra_ids]
    return df

ribo_counts = parse_fc(ribo_counts_file, "riboseq")
rnaseq_counts = parse_fc(rnaseq_counts_file, "rnaseq")

common_genes = ribo_counts.index.intersection(rnaseq_counts.index)
ribo_counts = ribo_counts.loc[common_genes]
rnaseq_counts = rnaseq_counts.loc[common_genes]

all_counts = pd.concat([ribo_counts, rnaseq_counts], axis=1)
all_counts = all_counts.loc[all_counts.sum(axis=1) > 0]
all_counts.index = [GENE_MAP.get(g, g) for g in all_counts.index]

print(f"Common genes with >0 reads: {len(all_counts):,}")
all_counts.head(10)

### Read count distributions

Overlaid histograms show the log10 read count distribution for each sample. Ribo-seq (blue tones) typically has a narrower, left-shifted distribution compared to RNA-seq (orange tones) because ribosome profiling captures a subset of the transcriptome.

In [ ]:
fig = go.Figure()
for i, col in enumerate(all_counts.columns):
    lc = np.log10(all_counts[col].replace(0,np.nan).dropna())
    fig.add_trace(go.Histogram(x=lc, name=col, opacity=0.5, nbinsx=80, marker_color=OKABE_ITO[i%len(OKABE_ITO)]))
fig.update_layout(title="Gene read count distribution", xaxis_title="log10(read count)", yaxis_title="Number of genes", barmode="overlay")
apply_publication_style(fig); set_legend_position(fig, "top-right")
fig.show()

### Top genes: Ribo-seq vs RNA-seq

**For your designs:** This chart directly shows which genes have the most ribosome occupancy. Genes where the blue bar (Ribo-seq) greatly exceeds the orange bar (RNA-seq) are being translated more actively than their mRNA levels would predict - their 3'UTRs may contain elements that enhance translation.

In [ ]:
riboseq_cols = [SAMPLES[s]["name"] for s in [s for s in SAMPLES if s in ribo_counts.columns and SAMPLES[s]["type"]=="riboseq"]]
rnaseq_cols  = [SAMPLES[s]["name"] for s in [s for s in SAMPLES if s in rnaseq_counts.columns and SAMPLES[s]["type"]=="rnaseq"]]

# Normalize per-sample to RPM before averaging
rpm = all_counts.copy()
for col in rpm.columns:
    rpm[col] = rpm[col] / rpm[col].sum() * 1e6
all_counts["Ribo-seq avg RPM"] = rpm[riboseq_cols].mean(axis=1)
all_counts["RNA-seq avg RPM"]  = rpm[rnaseq_cols].mean(axis=1)
all_counts["Ribo-seq avg raw"] = all_counts[riboseq_cols].mean(axis=1)
all_counts["RNA-seq avg raw"]  = all_counts[rnaseq_cols].mean(axis=1)

top = all_counts.nlargest(30, "Ribo-seq avg RPM")
fig = go.Figure()
fig.add_trace(go.Bar(name="Ribo-seq (RPM)", y=top.index, x=top["Ribo-seq avg RPM"], orientation="h", marker_color=TYPE_COLORS["riboseq"]))
fig.add_trace(go.Bar(name="RNA-seq (RPM)",  y=top.index, x=top["RNA-seq avg RPM"],  orientation="h", marker_color=TYPE_COLORS["rnaseq"]))
fig.update_layout(title="Top 30 genes by Ribo-seq RPM", xaxis_title="Average RPM", barmode="group")
apply_publication_style(fig, height=800); set_legend_position(fig, "bottom-right")
fig.show()

## 10. Translational Efficiency

**What:** Translational efficiency (TE) quantifies how actively an mRNA is being translated relative to its abundance:

$$TE = \\frac{\\text{Ribo-seq signal (CDS)}}{\\text{RNA-seq signal (exons)}}$$

**Why this matters for your designs:** TE reveals which mRNAs are translated efficiently and which are translationally repressed. The 3'UTRs of high-TE genes contain elements (sequence motifs, structures, lack of miRNA binding sites) that promote efficient translation. These are your best candidates for synthetic 3'UTR scaffolds.

**How:**
- Each sample is normalized using the median-of-ratios method (DESeq2 algorithm) to correct for sequencing depth and composition bias
- TE is computed in **log-space** (`log2(ribo + 1) - log2(rna + 1)`) to avoid asymmetric pseudocount artifacts
- TE is computed **per condition** (Tumor, Normal) to preserve biological signal

In [ ]:
# Median-of-ratios normalization (DESeq2 method, sparse-aware)
def median_of_ratios_normalize(count_df):
    log_counts = np.log(count_df.replace(0, np.nan))
    geo_means = np.exp(log_counts.mean(axis=1))
    valid = np.isfinite(geo_means) & (geo_means > 0)
    ratios = count_df.loc[valid].div(geo_means[valid], axis=0)
    size_factors = ratios.replace(0, np.nan).median(axis=0)
    normalized = count_df.div(size_factors, axis=1)
    print(f"  Size factors ({valid.sum()} genes): {dict(zip(count_df.columns, size_factors.round(2)))}")
    return normalized, size_factors

print("Ribo-seq normalization:")
ribo_norm, ribo_sf = median_of_ratios_normalize(ribo_counts)
print("RNA-seq normalization:")
rnaseq_norm, rnaseq_sf = median_of_ratios_normalize(rnaseq_counts)

### Per-condition TE computation

TE is computed separately for Tumor and Normal conditions. The minimum read count filter (`MIN_READS_TE=50`) removes genes with insufficient data for reliable ratio estimation.

In [ ]:
tumor_ribo = [c for c in ribo_norm.columns if "Tumor" in c]
normal_ribo = [c for c in ribo_norm.columns if "Normal" in c]
tumor_rna = [c for c in rnaseq_norm.columns if "Tumor" in c]
normal_rna = [c for c in rnaseq_norm.columns if "Normal" in c]

ribo_mean = ribo_counts.mean(axis=1)
rnaseq_mean = rnaseq_counts.mean(axis=1)
mask = (ribo_mean >= MIN_READS_TE) & (rnaseq_mean >= MIN_READS_TE)
print(f"Genes passing {MIN_READS_TE}-read filter: {mask.sum():,}")

te_results = pd.DataFrame(index=ribo_counts.index[mask])
te_results.index = [GENE_MAP.get(g, g) for g in te_results.index]

if tumor_ribo and tumor_rna:
    tr = ribo_norm.loc[mask, tumor_ribo].mean(axis=1).values
    tn = rnaseq_norm.loc[mask, tumor_rna].mean(axis=1).values
    te_results["log2_TE_tumor"] = np.log2(tr + 1) - np.log2(tn + 1)
    te_results["TE_tumor"] = 2 ** te_results["log2_TE_tumor"]

if normal_ribo and normal_rna:
    nr = ribo_norm.loc[mask, normal_ribo].mean(axis=1).values
    nn = rnaseq_norm.loc[mask, normal_rna].mean(axis=1).values
    te_results["log2_TE_normal"] = np.log2(nr + 1) - np.log2(nn + 1)
    te_results["TE_normal"] = 2 ** te_results["log2_TE_normal"]

te_results = te_results.replace([np.inf,-np.inf], np.nan).dropna()
te_cols = [c for c in te_results.columns if "log2_TE" in c]
te_results["log2_TE_mean"] = te_results[te_cols].mean(axis=1)

print(f"Genes with TE: {len(te_results):,}")
if "TE_tumor" in te_results.columns:
    print(f"Tumor median TE: {te_results['TE_tumor'].median():.2f}")
if "TE_normal" in te_results.columns:
    print(f"Normal median TE: {te_results['TE_normal'].median():.2f}")

### TE scatter plot

Each point is one gene. The x-axis shows RNA abundance (transcription), the y-axis shows ribosome occupancy (translation). Points above the diagonal have TE > 1 (translated more than expected from mRNA levels). The color scale (PuOr, colorblind-safe diverging) highlights the magnitude and direction of TE.

**For your designs:** Genes in the upper-left quadrant (high Ribo, low RNA) are the most translationally efficient. Their 3'UTRs are prime candidates for your synthetic constructs.

In [ ]:
ribo_rpm_viz = ribo_norm.loc[mask].mean(axis=1).values
rnaseq_rpm_viz = rnaseq_norm.loc[mask].mean(axis=1).values
scatter_df = pd.DataFrame({
    "log10_rna": np.log10(rnaseq_rpm_viz + 1), "log10_ribo": np.log10(ribo_rpm_viz + 1),
    "log2_TE": te_results["log2_TE_mean"].values, "gene": te_results.index,
})
fig = px.scatter(scatter_df, x="log10_rna", y="log10_ribo", color="log2_TE",
    color_continuous_scale="PuOr_r", color_continuous_midpoint=0, hover_data=["gene"],
    labels={"log10_rna":"log10(RNA-seq normalized)","log10_ribo":"log10(Ribo-seq normalized)","color":"log2(TE)"},
    title="Translational Efficiency")
rng = [min(scatter_df["log10_rna"].min(), scatter_df["log10_ribo"].min()),
       max(scatter_df["log10_rna"].max(), scatter_df["log10_ribo"].max())]
fig.add_trace(go.Scatter(x=rng, y=rng, mode="lines", line=dict(dash="dash",color="gray",width=1), name="TE = 1"))
fig.update_traces(marker=dict(size=4, opacity=0.6), selector=dict(mode="markers"))
apply_publication_style(fig, width=800, height=700)
fig.show()

### Differential TE: Tumor vs Normal (exploratory)

**Important caveat:** With n=1 Ribo-seq sample per condition, this analysis is purely descriptive. Individual genes cannot be statistically tested for differential TE. The ranking by |delta TE| identifies candidates for follow-up, not confirmed hits.

For formal differential TE testing, matched Ribo-seq/RNA-seq replicates and tools like Xtail or deltaTE are required.

In [ ]:
if "log2_TE_tumor" in te_results.columns and "log2_TE_normal" in te_results.columns:
    te_results["delta_log2_TE"] = te_results["log2_TE_tumor"] - te_results["log2_TE_normal"]
    te_results["abs_delta"] = te_results["delta_log2_TE"].abs()
    te_sorted = te_results.sort_values("abs_delta", ascending=False)

    print("Top 20 genes by |delta TE| (Tumor vs Normal, exploratory):")
    print("NOTE: Descriptive ranking only - n=1 Ribo-seq per condition.\n")
    print(te_sorted[["log2_TE_tumor","log2_TE_normal","delta_log2_TE"]].head(20))

    extreme = te_sorted.head(30)
    fig = go.Figure(go.Bar(y=extreme.index, x=extreme["delta_log2_TE"], orientation="h",
        marker_color=["#D55E00" if v>0 else "#009E73" for v in extreme["delta_log2_TE"]]))
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(title="Top 30 genes by |delta TE| (Tumor - Normal, exploratory)", xaxis_title="delta log2(TE)")
    apply_publication_style(fig, height=800)
    fig.show()
else:
    te_sorted = te_results.sort_values("log2_TE_mean")
    extreme = pd.concat([te_sorted.head(30), te_sorted.tail(30)])
    fig = go.Figure(go.Bar(y=extreme.index, x=extreme["log2_TE_mean"], orientation="h",
        marker_color=["#D55E00" if v<0 else "#009E73" for v in extreme["log2_TE_mean"]]))
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(title="Extreme TE genes (top/bottom 30)", xaxis_title="log2(TE)")
    apply_publication_style(fig, height=900)
    fig.show()

## 11. 3'UTR Coverage Heatmaps (deepTools + Plotly)

**What:** Visualize how ribosome density and mRNA coverage are distributed across 3'UTR regions for the most highly translated genes.

**Why this matters for your designs:** The 3'UTR is the primary region where you have design freedom in synthetic constructs (the CDS is dictated by the protein you want to make). By mapping exactly where ribosomes are on the 3'UTR, you can identify:
- **Read-through events**: Ribosomes that fail to stop at the stop codon and continue translating into the 3'UTR. These reveal leaky stop codon contexts you should avoid (or exploit).
- **Regulatory hotspots**: Positions where ribosome density drops sharply may correspond to regulatory elements (miRNA binding sites, AU-rich elements) that you could include or exclude in your designs.

**How:** deepTools `computeMatrix scale-regions` scales all 3'UTRs to the same length (1000 bins) with 200 bp flanks, then computes coverage from the bigWig tracks. This is orders of magnitude faster than per-gene pysam pileup.

Heatmaps use `Inferno` (Ribo-seq) and `Viridis` (RNA-seq) - perceptually uniform colormaps that are colorblind-safe and reproduce correctly in grayscale.

In [ ]:
utr_bed = BASE / "three_prime_utrs.bed"
if not utr_bed.exists():
    recs = []
    for line in open(GTF):
        if line.startswith("#"): continue
        f = line.strip().split("\t")
        if f[2]!="three_prime_utr": continue
        gid = re.search(r'gene_id "([^"]+)"', f[8])
        gn = re.search(r'gene_name "([^"]+)"', f[8])
        if gid:
            gi = gid.group(1).split(".")[0]
            recs.append((f[0], int(f[3])-1, int(f[4]), gn.group(1) if gn else gi, int(f[4])-int(f[3])+1, f[6], gi))
    udf = pd.DataFrame(recs, columns=["chrom","start","end","name","length","strand","gene_id"])
    udf = udf.sort_values("length",ascending=False).drop_duplicates("gene_id",keep="first").query("length>=200")
    te_gene_id_map = {GENE_MAP.get(g,g): g for g in ribo_counts.index[mask]}
    valid_ids = set(te_gene_id_map.values())
    udf = udf[udf["gene_id"].isin(valid_ids)]
    udf["score"] = udf["gene_id"].map(lambda g: ribo_counts.loc[g].mean() if g in ribo_counts.index else 0)
    udf = udf.nlargest(N_UTR_GENES, "score")
    udf[["chrom","start","end","name","score","strand"]].to_csv(utr_bed, sep="\t", header=False, index=False)
    print(f"Wrote {len(udf)} 3'UTR regions")
else: print("BED exists")

In [ ]:
matrix_gz = BASE / "utr_coverage_matrix.gz"
ribo_bws = [str(BIGWIG_DIR/f"{s}.bw") for s in SAMPLES if SAMPLES[s]["type"]=="riboseq" and (BIGWIG_DIR/f"{s}.bw").exists()]
rnaseq_bws = [str(BIGWIG_DIR/f"{s}.bw") for s in SAMPLES if SAMPLES[s]["type"]=="rnaseq" and (BIGWIG_DIR/f"{s}.bw").exists()]

if not matrix_gz.exists() and ribo_bws and rnaseq_bws:
    bws = [ribo_bws[0], rnaseq_bws[0]]
    print(f"Ribo: {Path(bws[0]).name} | RNA: {Path(bws[1]).name}")
    r = subprocess.run(["computeMatrix","scale-regions","-S"]+bws+["-R",str(utr_bed),
        "--regionBodyLength","1000","--beforeRegionStartLength","200","--afterRegionStartLength","200",
        "-o",str(matrix_gz),"--numberOfProcessors","8","--skipZeros"], capture_output=True, text=True)
    print("Done." if r.returncode==0 else f"ERROR: {r.stderr[-500:]}")
else: print("Matrix exists or bigWigs unavailable.")

### Parse and visualize coverage matrix

The deepTools matrix is parsed into separate Ribo-seq and RNA-seq arrays, row-normalized (each gene scaled to its own maximum), and sorted by read-through efficiency (RTE).

**RTE metric:** Rather than comparing first vs last portions of the UTR (which conflates different effects), we compute:

$$RTE = \\frac{\\text{3'UTR Ribo-seq} / \\text{CDS-flank Ribo-seq}}{\\text{3'UTR RNA-seq} / \\text{CDS-flank RNA-seq}}$$

This normalizes for mRNA structure effects: RTE > 1 means ribosomes are enriched in the 3'UTR beyond what mRNA levels would predict, a hallmark of stop codon read-through.

In [ ]:
with gzip.open(matrix_gz, "rt") as fh:
    hdr = json.loads(fh.readline().lstrip("@"))
    md_df = pd.read_csv(fh, sep="\t", header=None)
gene_names = md_df.iloc[:,3].values
sb = hdr.get("sample_boundaries",[])
if len(sb)>=3:
    bps = sb[1]-sb[0]
    ribo_mat = md_df.iloc[:,6:6+bps].values.astype(float)
    rnaseq_mat = md_df.iloc[:,6+bps:6+2*bps].values.astype(float)
else:
    h = (md_df.shape[1]-6)//2
    ribo_mat = md_df.iloc[:,6:6+h].values.astype(float)
    rnaseq_mat = md_df.iloc[:,6+h:].values.astype(float)
ribo_mat, rnaseq_mat = np.nan_to_num(ribo_mat), np.nan_to_num(rnaseq_mat)
print(f"Matrix: {ribo_mat.shape[0]} genes x {ribo_mat.shape[1]} bins")

In [ ]:
def row_norm(m):
    mx = m.max(axis=1, keepdims=True); mx[mx==0]=1; return m/mx
ribo_n, rna_n = row_norm(ribo_mat), row_norm(rnaseq_mat)

# CDS-normalized RTE
nc = ribo_mat.shape[1]
flank_bins = int(nc * 0.14)
utr_body_ribo = ribo_mat[:, flank_bins:].mean(axis=1)
cds_proxy_ribo = ribo_mat[:, :flank_bins].mean(axis=1)
utr_body_rna = rnaseq_mat[:, flank_bins:].mean(axis=1)
cds_proxy_rna = rnaseq_mat[:, :flank_bins].mean(axis=1)

valid_rt = (cds_proxy_ribo > 0) & (cds_proxy_rna > 0) & (utr_body_rna > 0) & (ribo_mat.sum(axis=1) > 0)
rt = np.full(len(cds_proxy_ribo), np.nan)
ribo_ratio = utr_body_ribo[valid_rt] / cds_proxy_ribo[valid_rt]
rna_ratio = utr_body_rna[valid_rt] / cds_proxy_rna[valid_rt]
rt[valid_rt] = ribo_ratio / rna_ratio

si = np.argsort(rt[valid_rt])[::-1]
rd = ribo_n[valid_rt][si]
rnd = rna_n[valid_rt][si]
nd = gene_names[valid_rt][si]
print(f"Genes with valid RTE: {valid_rt.sum()}")

In [ ]:
n = min(200, len(rd))
fig = make_subplots(rows=1, cols=2, subplot_titles=("Ribo-seq 3'UTR","RNA-seq 3'UTR"),
    shared_yaxes=True, horizontal_spacing=0.05)
fig.add_trace(go.Heatmap(z=rd[:n], y=nd[:n], colorscale="Inferno",
    colorbar=dict(title="Norm.", x=0.46, len=0.8)), row=1, col=1)
fig.add_trace(go.Heatmap(z=rnd[:n], y=nd[:n], colorscale="Viridis",
    colorbar=dict(title="Norm.", x=1.02, len=0.8)), row=1, col=2)
fig.update_layout(title="3'UTR coverage (sorted by read-through efficiency)",
    height=max(600,n*4), width=1200, plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=11, color="black"))
fig.update_xaxes(title_text="Position (-200 bp | scaled 3'UTR body | +200 bp)")
fig.update_yaxes(title_text="Gene", row=1, col=1, autorange="reversed")
fig.update_yaxes(row=1, col=2, autorange="reversed")
fig.show()

### Read-through candidates

Genes with RTE > 1 have more ribosome signal in the 3'UTR than expected from mRNA levels alone. These are candidates for stop codon read-through.

**For your designs:** Avoid the stop codon contexts of high-RTE genes if you want clean termination. Conversely, if you want to engineer read-through for polycistronic constructs, study the stop codon + immediate downstream sequence of these genes.

In [ ]:
rt_valid = rt[valid_rt][si]
rtdf = pd.DataFrame({"gene_name": nd, "RTE": rt_valid,
    "ribo_3utr_mean": ribo_mat[valid_rt][si].mean(1),
    "rnaseq_3utr_mean": rnaseq_mat[valid_rt][si].mean(1),
    "ribo_cds_mean": cds_proxy_ribo[valid_rt][si],
})
rtdf = rtdf[(rtdf["ribo_3utr_mean"]>1) & (rtdf["RTE"]>1)]
print(f"Read-through candidates (RTE > 1, top 20):")
rtdf.head(20)

## 12. 3'UTR Sequence Analysis for RNA Design

**What:** Extract the actual 3'UTR sequences from the top TE genes and analyze their sequence properties to identify features correlated with high translational efficiency.

**Why:** The preceding sections identify *which* genes have high TE and where ribosomes sit on 3'UTRs. This section answers the design question: **what sequence features make a good 3'UTR?** By comparing the 3'UTR sequences of high-TE vs low-TE genes, we can identify:
- Optimal 3'UTR length
- GC content preferences
- AU-rich element (ARE) density (AREs recruit RNA-binding proteins that can stabilize or destabilize mRNA)
- Potential miRNA binding sites (short seed matches to abundant miRNAs)

These features become design constraints for synthetic 3'UTRs.

In [ ]:
# Extract 3'UTR sequences for TE-ranked genes using pysam and the genome FASTA
import pysam

genome = pysam.FastaFile(str(GENOME_FA))

# Parse 3'UTR coordinates
utr_info = []
for line in open(GTF):
    if line.startswith("#"): continue
    f = line.strip().split("\t")
    if f[2] != "three_prime_utr": continue
    gid = re.search(r'gene_id "([^"]+)"', f[8])
    gn = re.search(r'gene_name "([^"]+)"', f[8])
    if gid:
        gi = gid.group(1).split(".")[0]
        utr_info.append({
            "chrom": f[0], "start": int(f[3])-1, "end": int(f[4]),
            "strand": f[6], "gene_id": gi,
            "gene_name": gn.group(1) if gn else gi,
            "length": int(f[4]) - int(f[3]) + 1,
        })

utr_info_df = pd.DataFrame(utr_info)
utr_info_df = utr_info_df.sort_values("length", ascending=False).drop_duplicates("gene_id", keep="first")
utr_info_df = utr_info_df[utr_info_df["length"] >= 50]

# Merge with TE data
te_gene_ids = {GENE_MAP.get(g,g): g for g in ribo_counts.index[mask]}
utr_info_df["gene_symbol"] = utr_info_df["gene_id"].map(GENE_MAP)
utr_info_df = utr_info_df[utr_info_df["gene_symbol"].isin(te_results.index)]
utr_info_df = utr_info_df.merge(
    te_results[["log2_TE_mean"]].rename(columns={"log2_TE_mean": "log2_TE"}),
    left_on="gene_symbol", right_index=True,
)

# Extract sequences
def get_seq(row):
    try:
        seq = genome.fetch(row["chrom"], row["start"], row["end"])
        if row["strand"] == "-":
            comp = str.maketrans("ACGTacgt", "TGCAtgca")
            seq = seq.translate(comp)[::-1]
        return seq.upper()
    except: return ""

utr_info_df["sequence"] = utr_info_df.apply(get_seq, axis=1)
utr_info_df = utr_info_df[utr_info_df["sequence"].str.len() > 0]

# Compute sequence features
utr_info_df["gc_pct"] = utr_info_df["sequence"].apply(lambda s: (s.count("G")+s.count("C"))/len(s)*100 if s else 0)
utr_info_df["au_pct"] = 100 - utr_info_df["gc_pct"]

# AU-rich elements (ARE): AUUUA pentamer, associated with mRNA instability
utr_info_df["are_count"] = utr_info_df["sequence"].apply(lambda s: s.replace("T","U").count("AUUUA"))
utr_info_df["are_density"] = utr_info_df["are_count"] / utr_info_df["length"] * 1000  # per kb

print(f"3'UTRs with TE data and sequences: {len(utr_info_df):,}")
genome.close()

### 3'UTR length vs translational efficiency

**Biological context:** Shorter 3'UTRs are generally associated with higher TE, possibly because they present fewer targets for miRNA-mediated repression and have lower probability of containing destabilizing elements. Many highly expressed housekeeping genes have short 3'UTRs.

**For your designs:** Consider using shorter 3'UTRs (100-500 nt) as a starting point for high-expression constructs.

In [ ]:
fig = px.scatter(utr_info_df, x="length", y="log2_TE", hover_data=["gene_name"],
    color="gc_pct", color_continuous_scale="Viridis",
    labels={"length": "3'UTR length (nt)", "log2_TE": "log2(TE)", "gc_pct": "GC (%)"},
    title="3'UTR length vs translational efficiency")
fig.update_traces(marker=dict(size=5, opacity=0.5))
fig.update_xaxes(type="log")
apply_publication_style(fig, width=900, height=600)
fig.show()

# Binned analysis
utr_info_df["length_bin"] = pd.cut(utr_info_df["length"], bins=[0,200,500,1000,2000,5000,50000],
    labels=["<200","200-500","500-1k","1k-2k","2k-5k",">5k"])
bin_stats = utr_info_df.groupby("length_bin")["log2_TE"].agg(["mean","std","count"]).reset_index()
print("\nMedian TE by 3'UTR length bin:")
for _, row in bin_stats.iterrows():
    print(f"  {row['length_bin']:>8s}: mean log2(TE) = {row['mean']:+.2f}, n = {int(row['count'])}")

### GC content and AU-rich elements vs TE

**Biological context:**
- **GC content**: High-GC 3'UTRs tend to form stable secondary structures that can impede ribosome scanning and translation. Low-GC (AU-rich) 3'UTRs are generally more accessible.
- **AU-rich elements (AREs)**: The pentamer AUUUA (and variants) recruits ARE-binding proteins (e.g., TTP, HuR) that regulate mRNA stability. High ARE density is associated with rapid mRNA turnover, which can reduce steady-state protein levels.

**For your designs:** Target moderate GC content (35-45%) and minimize AREs unless you specifically want tunable mRNA stability.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("GC content vs TE", "ARE density vs TE"))

fig.add_trace(go.Scatter(
    x=utr_info_df["gc_pct"], y=utr_info_df["log2_TE"],
    mode="markers", marker=dict(size=4, opacity=0.4, color="#56B4E9"),
    text=utr_info_df["gene_name"], name="genes",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=utr_info_df["are_density"], y=utr_info_df["log2_TE"],
    mode="markers", marker=dict(size=4, opacity=0.4, color="#E69F00"),
    text=utr_info_df["gene_name"], name="genes", showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="GC content (%)", row=1, col=1)
fig.update_xaxes(title_text="ARE density (AUUUA per kb)", row=1, col=2)
fig.update_yaxes(title_text="log2(TE)", row=1, col=1)
fig.update_yaxes(title_text="log2(TE)", row=1, col=2)
apply_publication_style(fig, width=1200, height=500)
fig.show()

# Correlation stats
r_gc, p_gc = stats.spearmanr(utr_info_df["gc_pct"], utr_info_df["log2_TE"])
r_are, p_are = stats.spearmanr(utr_info_df["are_density"], utr_info_df["log2_TE"])
print(f"GC vs TE: Spearman r = {r_gc:.3f}, p = {p_gc:.2e}")
print(f"ARE density vs TE: Spearman r = {r_are:.3f}, p = {p_are:.2e}")

## 13. 3'UTR Design Candidates

**What:** Rank 3'UTR sequences by their suitability as scaffolds for synthetic RNA constructs.

**Scoring criteria:**
1. **High TE** - the primary indicator of efficient translation
2. **Clean termination** - low RTE (no read-through), indicating a strong stop codon context
3. **Moderate length** - short enough to minimize regulatory complexity, long enough for stability
4. **Low ARE density** - fewer destabilizing elements
5. **Moderate GC** - avoids extreme secondary structures

The output is a ranked table of 3'UTR sequences ready for synthesis.

In [ ]:
# Merge TE and read-through data with sequence features
# Build RTE lookup from the heatmap analysis
rte_lookup = {}
if 'rtdf' in dir() and len(rtdf) > 0:
    for _, row in pd.DataFrame({"gene_name": gene_names[valid_rt][si], "RTE": rt[valid_rt][si]}).iterrows():
        rte_lookup[row["gene_name"]] = row["RTE"]

utr_info_df["RTE"] = utr_info_df["gene_name"].map(rte_lookup).fillna(1.0)
utr_info_df["clean_stop"] = utr_info_df["RTE"] < 1.0  # RTE < 1 means no read-through enrichment

# Design score: higher is better
# Weights: TE (primary), inverse length preference, low ARE, moderate GC, clean stop
utr_info_df["length_score"] = np.clip(1 - (utr_info_df["length"] - 300) / 5000, 0, 1)
utr_info_df["gc_score"] = 1 - np.abs(utr_info_df["gc_pct"] - 40) / 40  # peaks at 40% GC
utr_info_df["are_score"] = np.clip(1 - utr_info_df["are_density"] / 20, 0, 1)
utr_info_df["stop_score"] = (~(utr_info_df["RTE"] > 1.5)).astype(float)

# Normalize TE to 0-1 range
te_min, te_max = utr_info_df["log2_TE"].min(), utr_info_df["log2_TE"].max()
utr_info_df["te_score"] = (utr_info_df["log2_TE"] - te_min) / (te_max - te_min)

# Composite score (TE-weighted)
utr_info_df["design_score"] = (
    0.50 * utr_info_df["te_score"] +
    0.15 * utr_info_df["length_score"] +
    0.10 * utr_info_df["gc_score"] +
    0.10 * utr_info_df["are_score"] +
    0.15 * utr_info_df["stop_score"]
)

# Top candidates
candidates = utr_info_df.nlargest(50, "design_score")[
    ["gene_name","length","gc_pct","are_density","log2_TE","RTE","design_score","sequence"]
].reset_index(drop=True)

print(f"Top 20 3'UTR design candidates (of {len(utr_info_df):,} evaluated):\n")
print(candidates.drop(columns=["sequence"]).head(20).to_string())

### Export design candidates

The top 50 candidate 3'UTR sequences are exported as a FASTA file for downstream use in synthesis or further computational analysis (e.g., RNA structure prediction with ViennaRNA, codon optimization context).

In [ ]:
# Export top candidates as FASTA
fasta_path = BASE / "top_3utr_candidates.fasta"
with open(fasta_path, "w") as f:
    for _, row in candidates.iterrows():
        header = f">{row['gene_name']}_3UTR len={row['length']} GC={row['gc_pct']:.1f}% TE={row['log2_TE']:.2f} RTE={row['RTE']:.2f} score={row['design_score']:.3f}"
        f.write(f"{header}\n{row['sequence']}\n")
print(f"Exported {len(candidates)} candidate 3'UTRs to {fasta_path}")

# Also export full table as CSV
csv_path = BASE / "utr_design_analysis.csv"
utr_info_df.drop(columns=["sequence"]).to_csv(csv_path, index=False)
print(f"Full analysis table: {csv_path}")

## 14. Summary and Conclusions

### Key findings
- **Sample QC**: PCA and correlation analysis confirm expected grouping of Ribo-seq vs RNA-seq
- **Periodicity**: Reading frame distribution verifies genuine ribosome-protected fragments
- **Translational efficiency**: Per-condition TE analysis reveals genes with differential translation
- **Read-through**: CDS-normalized RTE identifies candidate stop codon read-through events
- **Design candidates**: Top 3'UTR sequences ranked by composite design score (TE, length, GC, ARE density, termination)

### Design rules extracted
1. **3'UTR length**: Shorter UTRs (200-500 nt) tend to have higher TE
2. **GC content**: Moderate GC (~35-45%) avoids inhibitory secondary structures
3. **AU-rich elements**: Low ARE density correlates with stable, efficiently translated mRNAs
4. **Stop codon context**: Genes with RTE < 1 have clean termination - study their stop codon + downstream sequence

### Limitations
- n=1 Ribo-seq per condition: differential TE is exploratory only
- No P-site offset correction (acceptable for gene-level and scaled heatmap analyses)
- Design score weights are heuristic; validate candidates experimentally

### Output files
- `top_3utr_candidates.fasta` - Top 50 3'UTR sequences for synthesis
- `utr_design_analysis.csv` - Full analysis table with all sequence features
- `multiqc_report/` - Aggregated QC report

### Methods
- Trimming: fastp | rRNA removal: bowtie2 (5S/5.8S/18S/28S) | Alignment: STAR 2.7.11b
- Counting: featureCounts (CDS for Ribo-seq, exon for RNA-seq)
- Normalization: median-of-ratios | TE: log-space, per-condition
- Coverage: deepTools bamCoverage + computeMatrix
- Visualization: Plotly (Okabe-Ito palette) | QC: MultiQC